# Sales Forecasting Pipeline — Orchestrator

Runs the full pipeline end to end:
ingest → prepare → find forecast level → ensemble model → decompose → export.

Edit `src/config.py` first (paths, column names, hierarchy) to match your data.

In [11]:
import sys

print(sys.executable)

c:\Users\mandavi\AppData\Local\Programs\Python\Python311\python.exe


In [12]:
import sys
sys.path.append("..")  # only needed if running from a subfolder

import pandas as pd
from src import config
from src.data_ingestion import DataIngestor
from src.data_preparation import DataPreparer, SparsityAnalyzer
from src.ensemble_model import EnsembleForecaster
from src.decomposition_engine import DecompositionEngine
from src.utils import get_logger

logger = get_logger("main", config.LOG_LEVEL)

## 1. Ingest raw data

In [13]:
ingestor = DataIngestor()
raw_df = ingestor.load()
raw_df.head()

2026-09-02 11:49:22 | src.data_ingestion | INFO | Loading data from C:\Users\mandavi\Downloads\sales_forecasting_pipeline\sales_forecasting_pipeline\data\sales_data.xlsx (detected format=xlsx, sheet=0)
2026-09-02 11:49:22 | src.data_ingestion | INFO | Loaded 1,001 rows, 10 columns.


,country,order_value_eur,cost,date,category,customer_name,sales_manager,sales_rep,device_type,order_id
0,Sweden,98320.37,77722.25,8/23/2020,Games,Konopelski LLC,Maxie Marrow,Tarrah Castelletti,Tablet,70-0511466
1,France,46296.26,40319.41,5/15/2020,Games,Wisoky Inc,Othello Bowes,Amelina Piscopiello,Tablet,77-3489084
2,Portugal,140337.34,115708.14,2020-04-09 00:00:00,Appliances,Hegmann Group,Celine Tumasian,Corene Shirer,PC,65-8218141
3,France,203604.46,175344.16,6/26/2019,Electronics,Kirlin and Sons,Othello Bowes,Crysta Halls,Mobile,29-5478106
4,UK,63979.04,56032.84,10/22/2019,Games,Schoen-Keeling,Jessamine Apark,Genevra Charrisson,PC,27-3437546


## 2. Prepare data & build the master table

In [14]:
preparer = DataPreparer()
clean_df = preparer.clean(raw_df)
master_df = preparer.build_master_table(clean_df)
master_df.head()

2026-09-02 11:49:22 | src.data_preparation | WARNING | Dropped 5 rows with invalid date/target values.
2026-09-02 11:49:23 | src.data_preparation | INFO | Master table built: 5,448 rows across 227 series.


,country,category,device_type,date,order_value_eur
0,Austria,Electronics,PC,2019-01-01,0.0
1,Austria,Games,PC,2019-01-01,0.0
2,Belgium,Accessories,PC,2019-01-01,0.0
3,Belgium,Appliances,Mobile,2019-01-01,0.0
4,Belgium,Clothing,Mobile,2019-01-01,0.0


## 3. Find the best forecast level

Walks the hierarchy to find the most granular level with enough dense history.

In [15]:
analyzer = SparsityAnalyzer()
forecast_level_cols, level_stats = analyzer.find_best_level(master_df)
print("Forecasting at level:", forecast_level_cols or ["TOTAL"])
level_stats.head()

2026-09-02 11:49:23 | src.data_preparation | INFO | Level ['country', 'category', 'device_type']: 227 series, 11% pass quality thresholds.
2026-09-02 11:49:23 | src.data_preparation | INFO | Level ['country', 'category']: 121 series, 25% pass quality thresholds.
2026-09-02 11:49:23 | src.data_preparation | INFO | Level ['country']: 15 series, 87% pass quality thresholds.
2026-09-02 11:49:23 | src.data_preparation | INFO | Selected forecast level: ['country']


Forecasting at level: ['country']


,country,non_zero_ratio,n_periods,passes
0,Austria,0.041667,24,False
1,Belgium,0.250000,24,False
2,Bulgaria,0.708333,24,True
3,Denmark,0.375000,24,True
4,Finland,0.958333,24,True


## 4. Aggregate to the chosen level and fit the ensemble

For simplicity this fits one ensemble on the total series at the chosen level.
For many independent series (e.g. per-region), loop this cell per group.

In [16]:
agg_df = (
    master_df.groupby([config.DATE_COL] + forecast_level_cols)[config.TARGET_COL]
    .sum()
    .reset_index()
)

# Collapse to a single series for the top-level ensemble fit (sum across the chosen level).
series = agg_df.groupby(config.DATE_COL)[config.TARGET_COL].sum()
series.index = pd.DatetimeIndex(series.index, freq=config.FREQUENCY)

ensemble = EnsembleForecaster()
ensemble.fit(series)

print("Backtest RMSE by model:", ensemble.backtest_errors)
print("Ensemble weights:", ensemble.weights)

Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
11:49:23 - cmdstanpy - INFO - Chain [1] start processing
11:49:43 - cmdstanpy - INFO - Chain [1] done processing
2026-09-02 11:49:43 | src.ensemble_model | INFO | Ensemble weights: {'arima': 0.5352921237446944, 'ets': 0.3554600117041679, 'prophet': 0.10924786455113775}
Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
11:49:44 - cmdstanpy - INFO - Chain [1] start processing
11:50:01 - cmdstanpy - INFO - Chain [1] done processing


Backtest RMSE by model: {'arima': 702650.0016516245, 'ets': 1058130.3079079394, 'prophet': 3442840.857156325}
Ensemble weights: {'arima': 0.5352921237446944, 'ets': 0.3554600117041679, 'prophet': 0.10924786455113775}


In [17]:
forecast_df = ensemble.predict(horizon=config.FORECAST_HORIZON)
forecast_df

,arima,ets,prophet,ensemble
2021-01-01,4.640851e+06,4.257369e+06,3.172570e+06,4.344132e+06
2021-02-01,4.681859e+06,3.659673e+06,2.168195e+06,4.043900e+06
2021-03-01,4.677731e+06,3.850065e+06,2.631541e+06,4.159987e+06
2021-04-01,4.678147e+06,5.047711e+06,3.400824e+06,4.669967e+06
2021-05-01,4.678105e+06,5.260752e+06,3.484590e+06,4.754824e+06
2021-06-01,4.678109e+06,5.512903e+06,3.964675e+06,4.896904e+06


## 5. Disaggregate the ensemble forecast to the granular hierarchy

In [18]:
decomposer = DecompositionEngine(hierarchy_cols=config.HIERARCHY_COLS)
decomposer.compute_shares(master_df, group_cols=config.HIERARCHY_COLS)

granular_forecast = decomposer.disaggregate(
    forecast_df[["ensemble"]], group_cols=config.HIERARCHY_COLS
)
granular_forecast.head()

,date,country,category,device_type,forecast,share
0,2021-01-01,Austria,Electronics,PC,8093.767838,0.001863
1,2021-01-01,Austria,Games,PC,22032.272321,0.005072
2,2021-01-01,Belgium,Accessories,PC,0.000000,0.000000
3,2021-01-01,Belgium,Appliances,Mobile,0.000000,0.000000
4,2021-01-01,Belgium,Clothing,Mobile,0.000000,0.000000


## 6. Export results

In [19]:
granular_forecast.to_csv(config.FORECAST_OUTPUT_FILE, index=False)

metrics_df = pd.DataFrame(
    {"model": list(ensemble.backtest_errors.keys()),
     "backtest_rmse": list(ensemble.backtest_errors.values())}
)
metrics_df["weight"] = metrics_df["model"].map(ensemble.weights).fillna(0)
metrics_df.to_csv(config.MODEL_METRICS_FILE, index=False)

print(f"Forecast written to {config.FORECAST_OUTPUT_FILE}")
print(f"Model metrics written to {config.MODEL_METRICS_FILE}")

Forecast written to C:\Users\mandavi\Downloads\sales_forecasting_pipeline\sales_forecasting_pipeline\outputs\final_forecast.csv
Model metrics written to C:\Users\mandavi\Downloads\sales_forecasting_pipeline\sales_forecasting_pipeline\outputs\model_metrics.csv
